# QLoRA fine-tuning on Colab -> push to the Hugging Face Hub

Fine-tunes a **small** model on `data/train.jsonl` and pushes the merged result to
your HF account, so the Vercel app can serve it via the `hf` backend.

**Setup:** Runtime -> Change runtime type -> **T4 GPU**.

**Steps:** (1) run the install cell, (2) set your HF **write** token + repo name,
(3) upload `data/train.jsonl` via the Files panel on the left, (4) run the rest.
At the end you get a repo id to paste into `FINETUNED_MODEL` in Vercel.

In [ ]:
%pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
%pip install -q --no-deps trl peft accelerate bitsandbytes

In [ ]:
from huggingface_hub import login

HF_TOKEN = ''  # a WRITE token from huggingface.co/settings/tokens
HF_REPO = 'your-username/rag-comparison-ft'  # where the fine-tuned model is pushed
# Small model = fast training + servable on HF. Llama needs licence acceptance;
# swap to 'unsloth/Qwen2.5-3B-Instruct-bnb-4bit' (ungated) if you prefer no gate.
MODEL = 'unsloth/Llama-3.2-3B-Instruct-bnb-4bit'
TRAIN_FILE = 'train.jsonl'  # upload data/train.jsonl into the Files panel
login(HF_TOKEN)

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL, max_seq_length=2048, load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=16, lora_dropout=0,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    use_gradient_checkpointing='unsloth',
)

In [ ]:
import json
from datasets import Dataset

rows = [json.loads(l) for l in open(TRAIN_FILE) if l.strip()]
def fmt(r):
    msgs = [{'role':'user','content':r['question']}, {'role':'assistant','content':r['answer']}]
    return {'text': tokenizer.apply_chat_template(msgs, tokenize=False)}
ds = Dataset.from_list([fmt(r) for r in rows])
print(f'{len(ds)} training examples')
print(ds[0]['text'][:300])

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=ds, dataset_text_field='text',
    args=SFTConfig(
        per_device_train_batch_size=2, gradient_accumulation_steps=4,
        warmup_steps=5, num_train_epochs=5, learning_rate=2e-4,
        logging_steps=1, optim='adamw_8bit', output_dir='outputs',
    ),
)
trainer.train()

## Push the merged model to the Hub

This uploads a merged (fp16) model so Hugging Face Inference can serve it. After it
finishes, set `FINETUNED_MODEL = HF_REPO` in your Vercel env vars.

In [ ]:
model.push_to_hub_merged(HF_REPO, tokenizer, save_method='merged_16bit', token=HF_TOKEN)
print('Done. Set FINETUNED_MODEL to:', HF_REPO)